# 06 - Démonstration sur des images inconnues

Le système complet tel qu'il tournerait à bord : **le fichier `modeles/one_look_int8.onnx` (0,66 Mo)
et les seuils de `modeles/config.json`**, rien d'autre. Pas de PyTorch, pas de GPU, pas d'annotation.

Images utilisées : pays **jamais vus à l'entraînement ni pour régler les seuils** (Bolivie, Espagne,
Ghana), puis des anomalies simulées. À la fin, une petite application permet de déposer n'importe
quelle image Sentinel-2 (GeoTIFF à 13 bandes).

Exécution : Colab, *Exécution > Tout exécuter* (environ 2 minutes, pas besoin de GPU).

In [ ]:
import os, subprocess, sys

if not os.path.isdir("one-look"):
    subprocess.run(["git", "clone", "-q", "https://github.com/lemembre05/one-look.git"], check=True)
else:
    subprocess.run(["git", "-C", "one-look", "pull", "-q"], check=True)
os.chdir("one-look")
sys.path.insert(0, ".")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio", "onnxruntime", "gradio"], check=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import demo, donnees

assert os.path.exists("modeles/config.json"), "modèle absent : lancer d'abord le notebook 05"
systeme = demo.OneLook("modeles")
print("Modèle :", systeme.chemin_modele, f"({os.path.getsize(systeme.chemin_modele) / 1e6:.2f} Mo)")
print("Seuils :", systeme.seuils)

## 1. Récupérer quelques images de test

Images Sentinel-2 brutes et carte d'eau permanente (connue à l'avance, c'est une carte fixe).
Les annotations ne servent **qu'à vérifier** après coup : le système ne les voit jamais.

In [ ]:
EXEMPLES = ["Bolivia_103757", "Bolivia_360519", "Spain_7370579", "Ghana_313799", "Ghana_866994", "Ghana_26376"]
BUCKET = donnees.BUCKET
os.makedirs("exemples", exist_ok=True)
for nom in EXEMPLES:
    for dossier in ["S2Hand", "JRCWaterHand", "LabelHand"]:
        cible = f"exemples/{nom}_{dossier}.tif"
        if not os.path.exists(cible):
            subprocess.run(["gsutil", "-q", "cp", f"{BUCKET}/{dossier}/{nom}_{dossier}.tif", cible])
EXEMPLES = [n for n in EXEMPLES if os.path.exists(f"exemples/{n}_S2Hand.tif")]
print(len(EXEMPLES), "images :", EXEMPLES)

## 2. Le système face à des images jamais vues

In [ ]:
lignes = []
for nom in EXEMPLES:
    s2 = demo.lire_tif(f"exemples/{nom}_S2Hand.tif")
    perm = demo.lire_tif(f"exemples/{nom}_JRCWaterHand.tif")[0]
    label = demo.lire_tif(f"exemples/{nom}_LabelHand.tif")[0]
    res = systeme.analyser(s2, perm)
    vrai = donnees.part_inondee_vraie(label, perm == 1)
    demo.figure(s2, res, nom); plt.show()
    print(demo.resume(res))
    if np.isnan(vrai):
        print("Vérité : image non annotée (presque entièrement nuageuse)\n")
    else:
        print(f"Vérité (annotation) : {vrai:.1%} inondé → {'alerte attendue' if vrai >= 0.05 else 'pas d’alerte attendue'}\n")
    lignes.append({"image": nom, "decision": res["decision"], "eau_detectee": round(res["part_eau"], 3),
                   "inonde_vrai": round(float(vrai), 3), "nuages": round(res["nuages"], 2),
                   "temps_ms": round(res["temps_ms"])})
pd.DataFrame(lignes)

## 3. Anomalies simulées

Un capteur qui sature, des bandes infrarouges perdues, du bruit : le système doit refuser de
conclure (**INCONNU**) au lieu de lancer une fausse alerte.

In [ ]:
nom = EXEMPLES[0]
s2 = demo.lire_tif(f"exemples/{nom}_S2Hand.tif")
perm = demo.lire_tif(f"exemples/{nom}_JRCWaterHand.tif")[0]
anomalies = {
    "originale": s2,
    "capteur saturé (× 3)": s2 * 3,
    "bandes infrarouges perdues": np.concatenate([s2[:7], np.zeros_like(s2[7:])]),
    "bruit": np.random.default_rng(0).uniform(0, 10000, s2.shape).astype(np.float32),
}
lignes = []
for t, x in anomalies.items():
    r = systeme.analyser(x, perm)
    lignes.append({"transformation": t, "decision": r["decision"], "score_inconnu": round(r["score_inconnu"], 1)})
print(f"Image de départ : {nom} ; seuil d'inconnu : {systeme.seuils['inconnu']}")
pd.DataFrame(lignes)

## 4. Temps de calcul sur un seul cœur de processeur

Image complète 512 × 512 pixels, réseau + décision.

In [ ]:
import time
x = demo.lire_tif(f"exemples/{EXEMPLES[0]}_S2Hand.tif")
systeme.analyser(x)  # échauffement
t = [systeme.analyser(x)["temps_ms"] for _ in range(5)]
print(f"{np.mean(t):.0f} ms par image (min {np.min(t):.0f}, max {np.max(t):.0f})")

## 5. Application de démonstration

Déposer une image Sentinel-2 (GeoTIFF à 13 bandes, niveau L1C, comme Sen1Floods11), éventuellement
la carte d'eau permanente, et simuler une anomalie. Les images de test ci-dessus sont proposées en
exemples.

In [ ]:
import runpy
app = runpy.run_path("app/app.py", run_name="demo")["app"]
app.launch(share=False, inline=True, height=850)